In [ ]:
# --- Setup: make the `ecp` support package available -----------------
# Colab opens a single notebook and installs nothing, so fetch `ecp` from
# the public repo if it isn't importable yet. On Binder/local it is already
# installed, so this cell is a fast no-op there.
try:
    import ecp  # noqa: F401
except ModuleNotFoundError:
    import subprocess, sys
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q",
         "git+https://github.com/ramador09/elementary-computational-physics-binder@main"],
        check=True,
    )


# 7.5 The Quantum Oscillator at Temperature: Planck's Occupation, Freezing Out, and the Classical Limit

<!-- This single H1 (one per notebook, "# <number> <Title>") is the page's
     title: it sets the sidebar entry, breadcrumb, browser tab, and search
     result. The branded banner below is generated by the shared `ecp`
     package, so the look of every notebook in the series lives in one place. -->

In [ ]:
from ecp.style import header, use_style

use_style()  # apply the series Matplotlib style
header(
    volume="Volume VII — Quantum Statistical Mechanics",
    number="7.5",
    title="The Quantum Oscillator at Temperature: Planck's Occupation, Freezing Out, and the Classical Limit",
    blurb="The calculation quantum mechanics was born from, done exactly. One warm "
    "oscillator yields the Planck occupation — the Bose function, one movement "
    "early — a heat capacity that freezes below a characteristic temperature and "
    "turns classical above it, and a high-temperature limit so sharp it derives the "
    "h that classical statistical mechanics had to insert by hand. Einstein's solid "
    "then closes a nineteenth-century scandal to three digits, and fails at low "
    "temperature in exactly the way that summons Debye.",
    difficulty="advanced",
    estimate="185–225 min",
)

## Notebook overview

This notebook does the volume's founding calculation: the one Planck performed in 1900 to fit the
blackbody spectrum, and the one Einstein pointed at solid matter in 1907. The system is the
harmonic oscillator of [§6.12](../06-quantum-mechanics/harmonic-oscillator.ipynb) (the ladder spectrum $E_n = \hbar\omega(n + \tfrac12)$), placed in
contact with a heat bath by the machinery of [§7.4](thermal-density-matrix.ipynb). It is *exactly* solvable: the partition function is
a geometric series, so every claim made here is verified to machine precision rather than
estimated, and one warm oscillator turns out to contain a remarkable fraction of the volume in
miniature.

The haul, in order. The exact $Z$ delivers $\langle E\rangle = \hbar\omega(\langle n\rangle +
\tfrac12)$ with the **Planck occupation** $\langle n\rangle = 1/(e^{\beta\hbar\omega}-1)$ — and
the reader should recognize it: this is the **Bose–Einstein function** at $\mu = 0$, appearing
one movement before [§7.7](bose-einstein-fermi-dirac.ipynb) derives it from the grand canonical ensemble. No accident: a bosonic
mode *is* a harmonic oscillator, its quanta the rungs of the ladder. Movement I thereby completes
a quiet symmetry — the warm qubit of [§7.4](thermal-density-matrix.ipynb) produced the Fermi function, this notebook's warm oscillator
produces the Bose function, each from a single exactly-solvable system, and the ensemble
derivation of [§7.7](bose-einstein-fermi-dirac.ipynb) (plus the Matsubara contours of [§7.2](complex-analysis-applications.ipynb)) must land on the same two functions. The
heat capacity then tells the volume's second "where classical failed" story quantitatively: below
the **Einstein temperature** $\theta_E = \hbar\omega/k_B$ the oscillator **freezes out** —
equipartition's guaranteed $k_B$ simply switches off — and above it the classical answer returns.
The classical limit is made honest twice over: the high-temperature limit of the quantum $Z$
*derives* the $1/h$ that Volume V inserted into its phase-space measure by fiat, and the leading
quantum correction $(\hbar\omega)^2/12k_BT$ is measured converging — the first term of the Wigner
expansion. Scaled up to $3N$ oscillators, the **Einstein solid** closes the nineteenth century's
diamond scandal to three digits and then fails, honestly, in exactly the way that makes Debye
([§7.16](phonons-debye.ipynb)) necessary.

Two payload exercises plant seeds. The oscillator's thermal number distribution is *geometric*,
and two million samples of it exhibit **bunching** — the super-Poissonian variance
$\langle\Delta n^2\rangle = \langle n\rangle(1 + \langle n\rangle)$ whose $+\langle n\rangle^2$
excess Einstein read, in 1909, as the wave half of wave–particle duality: the photon gas of [§7.14](photon-gas-planck.ipynb)
begins here. And the **thermal width** $\langle x^2\rangle = (\hbar/2m\omega)\coth(\beta\hbar
\omega/2)$, computed by ladder-operator trace machinery, traces the full arc from the pure
ground-state spread (a quantum width that never freezes) to classical equipartition growth —
flagged now as the exact curve that path-integral Monte Carlo ([§7.21](path-integral-monte-carlo.ipynb)) will re-derive by sampling
classical ring polymers. Molecules are deliberately deferred to [§7.6](molecules-rotation-vibration.ipynb), coupled oscillators to [§7.16](phonons-debye.ipynb).

> **Conventions (this notebook).** Working units $\hbar = \omega = k_B = 1$ (so $\beta = 1/T$,
> the level spacing is $1$, and $\theta_E = 1$), with physical units restored where the physics
> demands them — the Einstein-solid section speaks Kelvin and J mol⁻¹K⁻¹. Truncated spectral sums
> always state their $n_{\max}$, governed by the rule $n_{\max} \gg k_BT/\hbar\omega$: the
> truncation trap bites at *high* temperature, not low. Planck denominators are evaluated with
> `numpy.expm1` to dodge catastrophic cancellation at small $\beta\hbar\omega$. Random sampling
> uses a fixed seed.
>
> **How to read the checks.** Each exercise closes with a `validate` call against an independent
> fact: the geometric-series $Z$ against brute-force truncated sums (and the stated failure of an
> undersized $n_{\max}$); $-\partial\ln Z/\partial\beta$ against $\hbar\omega(n_B + \tfrac12)$ by
> central differences; the frozen and classical heat-capacity values at $0.1\,\theta_E$ and
> $50\,\theta_E$ with the exponential law fitted on a log axis; the $1/h$ measure derived from
> the high-$T$ ratio and the Wigner correction converging to $1$; the diamond number against the
> measured $0.245$; two million geometric samples against the exact mean and bunched variance
> (cross-checked by $\partial^2\ln Z/\partial\beta^2$); and the ladder-trace $\langle x^2\rangle$
> against the $\coth$ closed form at three temperatures. A ✓ is strong evidence; a ✗ is a prompt
> to *locate the discrepancy*.
>
> **Scope.** One oscillator (and $3N$ independent copies of it), exactly. Molecules and their
> staircase are [§7.6](molecules-rotation-vibration.ipynb); the ensemble derivations of $n_B$/$n_F$ are [§7.7](bose-einstein-fermi-dirac.ipynb); the photon *gas* is [§7.14](photon-gas-planck.ipynb);
> coupled oscillators and the Debye spectrum are [§7.16](phonons-debye.ipynb) (motivated here, not built); the
> path-integral resampling of today's $\langle x^2\rangle$ is [§7.21](path-integral-monte-carlo.ipynb). See Pathria & Beale (Ch. 7);
> Kittel, *Introduction to Solid State Physics* (the Einstein and Debye models and the diamond
> data); Planck (1900) and Einstein (1907), the historical seeds; Mandel & Wolf on photon
> statistics. Cross-reference [§6.12](../06-quantum-mechanics/harmonic-oscillator.ipynb) (the ladder spectrum and operators, reused wholesale), [§7.4](thermal-density-matrix.ipynb) (the
> canonical identities and the $n_F$ twin), [§7.2](complex-analysis-applications.ipynb) (the Matsubara route), [§5.5](../05-classical-stat-mech/ergodicity.ipynb)/[§5.6](../05-classical-stat-mech/ideal-gas-fundamental-relation.ipynb) (equipartition and
> the classical measure, about to be derived).

## Theory in brief

### The exact partition function

The oscillator's spectrum is the ladder $E_n = \hbar\omega(n + \tfrac12)$, $n = 0, 1, 2, \dots$
([§6.12](../06-quantum-mechanics/harmonic-oscillator.ipynb)), so the canonical sum is geometric:

```{math}
:label: eq-oscillator-Z
Z = \sum_{n=0}^{\infty} e^{-\beta\hbar\omega(n+\frac12)}
= \frac{e^{-\beta\hbar\omega/2}}{1 - e^{-\beta\hbar\omega}}
= \frac{1}{2\sinh(\beta\hbar\omega/2)} .
```

The first equality pulls out the zero-point factor and sums the geometric series in
$e^{-\beta\hbar\omega}$; the $\sinh$ form is the same expression folded symmetrically. Because the
series is infinite, any *truncated* check must respect the convergence rule $n_{\max} \gg
k_BT/\hbar\omega$ — the occupied rungs reach up to $n \sim k_BT/\hbar\omega$, so the truncation
trap bites at **high** temperature, where intuition least expects it. Free energy and entropy
follow from the identities of [§7.4](thermal-density-matrix.ipynb), and $S \to 0$ as $T \to 0$: the third law holds on an infinite
spectrum exactly as it did on two levels.

### The Planck occupation, and the movement's symmetry

One derivative generates the physics:

```{math}
:label: eq-planck-occupation
\langle E\rangle = -\frac{\partial\ln Z}{\partial\beta}
= \hbar\omega\left(\langle n\rangle + \tfrac12\right),
\qquad
\langle n\rangle = \frac{1}{e^{\beta\hbar\omega} - 1} .
```

The $\tfrac12$ is the zero-point energy of [§6.12](../06-quantum-mechanics/harmonic-oscillator.ipynb), now a *thermodynamic* floor that survives at $T = 0$.
And $\langle n\rangle$ deserves a long look: it is the **Bose–Einstein function** $n_B(\hbar
\omega)$ at $\mu = 0$, one movement before its official derivation. The reason is structural,
not coincidental: a bosonic *mode* is a harmonic oscillator, and the number of quanta on the
ladder is the mode's occupation. Movement I has now produced both quantum statistics from single
systems (the two-level qubit gave $n_F$ in [§7.4](thermal-density-matrix.ipynb), the oscillator gives $n_B$), and when [§7.7](bose-einstein-fermi-dirac.ipynb) derives
both from the grand canonical ensemble (and the Matsubara contours of [§7.2](complex-analysis-applications.ipynb) are recalled), three
independent routes will have converged on the same pair of functions.

### Freezing out: where equipartition dies

Differentiating $\langle E\rangle(T)$ gives the oscillator's heat capacity,

```{math}
:label: eq-freezing-out
C(T) = k_B\,\frac{x^2 e^{x}}{(e^{x}-1)^2},
\qquad
x = \beta\hbar\omega = \frac{\theta_E}{T},
\qquad
\theta_E \equiv \frac{\hbar\omega}{k_B},
```

and the **Einstein temperature** $\theta_E$ splits the world in two. Below it the oscillator is
**frozen**: $k_BT$ cannot reach the first excited state, the Boltzmann factor strangles every
excitation, and $C \approx k_B x^2 e^{-x}$ dives exponentially. Above it, equipartition returns:
$C \to k_B$, Volume V's answer. This is the volume's second "where classical failed" resolution:
classical equipartition ([§5.5](../05-classical-stat-mech/ergodicity.ipynb)/[§5.6](../05-classical-stat-mech/ideal-gas-fundamental-relation.ipynb)) assigns $k_B$ per oscillator *unconditionally*, and experiment
said otherwise; quantum discreteness switches degrees of freedom **off** when $k_BT$ sinks below
their quantum. The single number $\theta_E$ organizes this notebook and the two after it.

### The classical limit, made honest twice

At high temperature the quantum partition function approaches

```{math}
:label: eq-classical-limit
Z_q \;\xrightarrow{\;\beta\hbar\omega\to0\;}\; \frac{k_BT}{\hbar\omega},
\qquad\text{while}\qquad
\int \frac{dp\,dx}{h}\,e^{-\beta H_{cl}} = \frac{2\pi k_BT/\omega}{h} = \frac{k_BT}{\hbar\omega} .
```

Volume V's classical oscillator integral $\int dp\,dx\,e^{-\beta H}$ equals $2\pi k_BT/\omega$ —
a quantity with *dimensions of action*, which cannot be a state count as it stands. It becomes
one, and matches the quantum limit, only when divided by $h = 2\pi\hbar$: the $1/h$ that Volume V
inserted by convention is thereby **derived** as the unique normalization under which classical
statistical mechanics is the high-temperature limit of quantum statistical mechanics. (The
measure's other mystery factor, the $1/N!$, is derived the same way in [§7.8](classical-limit-thermal-wavelength.ipynb).) The *approach* to the
limit is itself measurable: expanding $x/(e^x-1)$ gives $\langle E\rangle = k_BT +
(\hbar\omega)^2/12k_BT + \dots$, and the normalized correction $(\langle E\rangle - k_BT)\cdot
12k_BT/(\hbar\omega)^2 \to 1$ — the leading term of the **Wigner expansion**, the systematic
$\hbar^2$ bookkeeping of the quantum–classical borderland.

### The Einstein solid and the diamond anomaly

Einstein's 1907 move: model a crystal of $N$ atoms as $3N$ independent oscillators at one
frequency $\omega_E$,

```{math}
:label: eq-einstein-solid
\frac{C_{\text{solid}}}{3Nk_B} = \frac{x^2 e^{x}}{(e^{x}-1)^2},
\qquad
x = \frac{\theta_E}{T} .
```

At high temperature this is the **Dulong–Petit law** $C = 3Nk_B \approx 24.9$ J mol⁻¹K⁻¹, which
nineteenth-century calorimetry confirmed for most solids at room temperature. **Diamond** was the
scandal: its room-temperature heat capacity is a quarter of the law's value. Einstein's curve
resolves it: light atoms and stiff bonds give diamond $\theta_E \approx 1320$ K, so room
temperature sits *deep in the frozen regime*, and the model predicts $C/3Nk_B = 0.244$ at 300 K
against the measured $\approx 0.245$ — a sixty-year-old anomaly closed to three digits. The model
also fails honestly: at low $T$ its exponential tail collapses far below the *observed* $T^3$
law, because independent equal-frequency oscillators all share one gap, while a real crystal's
coupled modes include arbitrarily soft long-wavelength sound that no temperature can freeze.
That discrepancy is Debye's cue ([§7.16](phonons-debye.ipynb)) — motivated here, not merely announced.

### Thermal photon statistics: the geometric distribution and bunching

The thermal state's number distribution is the Boltzmann ladder itself,

```{math}
:label: eq-photon-statistics
p_n = \left(1 - e^{-x}\right)e^{-n x},
\qquad
\langle\Delta n^2\rangle = \langle n\rangle\left(1 + \langle n\rangle\right)
\;>\; \langle n\rangle_{\text{Poisson}},
```

a **geometric distribution** on $n \ge 0$ — samplable directly (with one off-by-one care point:
`numpy`'s geometric sampler counts trials from $1$, so $n = k - 1$). Its variance *exceeds* the
Poisson value $\langle n\rangle$ by $\langle n\rangle^2$: thermal light is **super-Poissonian**,
*bunched*. That excess is precisely the "wave" term of Einstein's 1909 fluctuation formula, and
the classical face of what Hanbury Brown and Twiss later measured in starlight. The contrast with
Poissonian laser light, and the full photon gas, are [§7.14](photon-gas-planck.ipynb) and the business of [§7.15](einstein-a-b-coefficients.ipynb) — this is the seed.

### The thermal width: the curve the path integral will resample

Averaging the eigenstate widths $\langle x^2\rangle_n = (n + \tfrac12)\hbar/m\omega$ over the
Boltzmann populations gives

```{math}
:label: eq-thermal-width
\langle x^2\rangle(T) = \frac{\hbar}{2m\omega}\coth\!\left(\frac{\beta\hbar\omega}{2}\right)
\;\longrightarrow\;
\begin{cases}
\hbar/2m\omega & T \to 0 \quad(\text{the pure ground-state spread})\\[4pt]
k_BT/m\omega^2 & T \to \infty \quad(\text{classical equipartition of } \tfrac12 m\omega^2 x^2),
\end{cases}
```

a single curve running from a quantum floor that never freezes to classical linear growth. We
compute it twice — the closed form, and the machine route $\operatorname{Tr}(\rho\,x^2)$ with the
ladder matrices of Volume VI — and flag it explicitly: this is the exact curve that
path-integral Monte Carlo ([§7.21](path-integral-monte-carlo.ipynb)) will re-derive by Metropolis sampling of classical ring
polymers, and the comparison will be that method's acid test.

## Setup

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from scipy.integrate import quad

from ecp import draw, validate

ACCENT, INK, SOFT = draw.ACCENT, draw.INK, draw.SOFT
RED = "#c1121f"

# Conventions: working units ħ = ω = k_B = m = 1 (level spacing 1,
# θ_E = 1, β = 1/T); the Einstein-solid exercise restores Kelvin and J/(mol·K).
# Truncated spectral sums STATE their n_max and obey n_max >> k_BT/ħω — the trap
# bites at HIGH temperature. Planck denominators use numpy.expm1 (e^x − 1 without
# cancellation at small x). Seeds fixed where sampling appears.


def Z_oscillator(beta, n_max=None):
    """The oscillator partition function: closed form, or truncated spectral sum (eq-oscillator-Z).

    With n_max None (default), returns the geometric-series closed form
    1/(2 sinh(β/2)) in working units ħω = 1. With n_max given, returns the brute-force
    truncated sum Σ_(n=0..n_max) exp(−β(n + 1/2)) — the check the closed form must
    survive. Convergence rule: the occupied rungs reach n ~ k_BT/ħω = 1/β, so demand
    n_max >> 1/β; at β = 0.2 an n_max of 60 already fails the eight-digit standard
    (Exercise 1 demonstrates it).

    Parameters
    ----------
    beta : float
        Inverse temperature (working units).
    n_max : int, optional
        Truncation of the spectral sum; None (default) selects the closed form.

    Returns
    -------
    float
        Z(β) by the requested route.
    """
    if n_max is None:
        return 1.0 / (2.0 * np.sinh(beta / 2.0))
    n = np.arange(0, n_max + 1, dtype=float)
    return float(np.sum(np.exp(-beta * (n + 0.5))))


def planck_occupation(beta):
    """The Planck occupation ⟨n⟩ = 1/(e^(βħω) − 1) — the Bose function at μ = 0.

    Evaluated as 1/expm1(β) (working units): numpy.expm1 computes e^x − 1 without the
    catastrophic cancellation that 'np.exp(x) − 1' suffers for small x, where the
    result is ~x and a naive subtraction loses most of its digits — the difference
    between a clean classical limit and numerical noise at β ~ 1e-8.

    Parameters
    ----------
    beta : float or numpy.ndarray
        Inverse temperature (working units ħω = 1).

    Returns
    -------
    float or numpy.ndarray
        The mean occupation ⟨n⟩.
    """
    return 1.0 / np.expm1(beta)


def heat_capacity(x):
    """The oscillator heat capacity C/k_B = x^2 e^x/(e^x − 1)^2 at x = βħω (eq-freezing-out).

    Written expm1-safely as x^2 (expm1(x) + 1)/expm1(x)^2, which keeps full accuracy in
    the classical regime x → 0 (where C/k_B → 1 − x^2/12 and the naive denominator
    cancels catastrophically) and underflows benignly to 0 in the frozen regime x >> 1.
    Doubles as the Einstein-solid curve per mode: C_solid/(3N k_B) is this function at
    x = θ_E/T.

    Parameters
    ----------
    x : float or numpy.ndarray
        The dimensionless gap-to-temperature ratio βħω = θ_E/T.

    Returns
    -------
    float or numpy.ndarray
        C/k_B per oscillator.
    """
    em = np.expm1(x)
    return x**2 * (em + 1.0) / em**2


def sample_photon_number(beta, size, rng):
    """Draw thermal (geometric) occupation numbers n ≥ 0 for one mode (eq-photon-statistics).

    numpy's rng.geometric(p) counts the TRIALS to first success, k = 1, 2, 3, …, while
    the thermal distribution p_n = (1 − e^(−β)) e^(−nβ) lives on n = 0, 1, 2, … —
    identical shapes, shifted by one. The mapping n = k − 1 is applied here, once,
    where it belongs; missing it silently inflates ⟨n⟩ by exactly 1 and is the classic
    off-by-one of thermal-photon sampling. The success probability is p = 1 − e^(−β),
    computed as −expm1(−β).

    Parameters
    ----------
    beta : float
        Inverse temperature (working units ħω = 1).
    size : int
        Number of samples.
    rng : numpy.random.Generator
        The seeded generator (numpy.random.default_rng).

    Returns
    -------
    numpy.ndarray
        Integer occupation samples n ≥ 0.
    """
    return rng.geometric(p=-np.expm1(-beta), size=size) - 1


def thermal_x2(beta, n_max=400):
    """⟨x^2⟩ at inverse temperature β by the ladder-operator trace route (eq-thermal-width).

    Volume VI's construction, reused verbatim: a = numpy.diag(sqrt(1..n_max), +1) is
    the annihilation matrix in the truncated number basis, x = (a + a†)/√2 in working
    units (ħ = m = ω = 1), and the thermal average is Tr(ρ x^2) with ρ the shifted
    Boltzmann populations on the ladder. Truncation is honest as long as the thermal
    occupation has died well below n_max — the default 400 covers β down to 0.1 (where
    ⟨n⟩ ≈ 10) with e^(−40)-level tails; only the single corner element ⟨n_max|x^2|n_max⟩
    is wrong, and it carries exponentially negligible weight.

    Parameters
    ----------
    beta : float
        Inverse temperature (working units).
    n_max : int, optional
        Number-basis truncation (default 400).

    Returns
    -------
    float
        The thermal width ⟨x^2⟩ = Tr(ρ x^2).
    """
    n = np.arange(1, n_max + 1, dtype=float)
    a = np.diag(np.sqrt(n), 1)  # annihilation: a|n⟩ = √n |n−1⟩
    x_op = (a + a.T) / np.sqrt(2.0)
    x2 = x_op @ x_op
    E_n = np.arange(0, n_max + 1, dtype=float) + 0.5
    w = -beta * E_n
    p = np.exp(w - w.max())
    p /= p.sum()
    return float(np.trace(np.diag(p) @ x2).real)

## Exercise 1 — The exact partition function

Planck's geometric series — the volume's founding calculation, done in closed form and checked
against brute force, with the truncation trap demonstrated where it actually bites. Cite
{eq}`eq-oscillator-Z`.

1. Derive $Z = e^{-\beta\hbar\omega/2}/(1-e^{-\beta\hbar\omega}) = 1/(2\sinh(\beta\hbar\omega/2))$
   by summing the geometric series over $E_n = \hbar\omega(n+\tfrac12)$.
2. Implement both routes in `Z_oscillator`, the closed form and the truncated spectral sum, and
   confirm agreement to at least eight digits at $\beta = 2$ ($n_{\max} = 60$) and $\beta = 0.2$
   ($n_{\max} = 400$).
3. State and demonstrate the truncation rule: convergence requires $n_{\max} \gg k_BT/\hbar\omega$,
   so the trap bites at *high* temperature — show $n_{\max} = 60$ failing the eight-digit standard
   at $\beta = 0.2$.
4. Compute $F(T)$ and $S(T)$ via the identities of [§7.4](thermal-density-matrix.ipynb) and confirm $S \to 0$ as $T \to 0$: the third
   law holding on an infinite spectrum, not just the two levels of [§7.4](thermal-density-matrix.ipynb).

In [ ]:
# (solution hidden on the public site)


### Validation 1

In [ ]:
validate.close(
    [Z_closed_cold, Z_closed_hot],
    [Z_trunc_cold, Z_trunc_hot],
    "the oscillator partition function: geometric series, verified by brute force at both regimes",
    rtol=1e-8,
)
validate.check(
    trunc_err_short > 1e-8 and trunc_err_long < 1e-10,
    "the truncation rule n_max >> k_BT/ħω: n_max = 60 fails at β = 0.2, n_max = 400 does not",
    f"errors {trunc_err_short:.1e} vs {trunc_err_long:.1e}",
)
validate.check(
    S_vals[0] < 1e-6
    and abs(F_vals[0] - 0.5) < 1e-6
    and bool(np.all(np.diff(S_vals) > 0)),
    "S → 0 and F → E_0 = ħω/2 as T → 0: the third law on an infinite spectrum",
    f"S(0.05) = {S_vals[0]:.1e}, F(0.05) = {F_vals[0]:.6f}",
)

## Exercise 2 — The Planck occupation: the Bose function, one movement early

One derivative of $\ln Z$, and the second of the two quantum statistics appears. Cite
{eq}`eq-planck-occupation`.

1. Compute $\langle E\rangle = -\partial\ln Z/\partial\beta$ by central differences (step
   $10^{-6}$) and confirm it equals $\hbar\omega(\langle n\rangle + \tfrac12)$ with
   $\langle n\rangle = 1/(e^{\beta\hbar\omega}-1)$ (the `planck_occupation` helper, `numpy.expm1`
   inside) to at least eight digits.
2. Identify the two pieces: the zero-point floor $\hbar\omega/2$ — present at $T = 0$, the
   ground energy of [§6.12](../06-quantum-mechanics/harmonic-oscillator.ipynb) now thermodynamic — and the thermal occupation $\langle n\rangle$.
3. Recognize $\langle n\rangle$ as the Bose–Einstein function $n_B(\hbar\omega)$ at $\mu = 0$,
   and explain why: a bosonic mode *is* an oscillator, its quanta the rungs.
4. Pair with [§7.4](thermal-density-matrix.ipynb) in prose: the qubit produced $n_F$, the oscillator produces $n_B$ — both
   statistics have now appeared from single exactly-solvable systems, and the ensemble
   derivation of [§7.7](bose-einstein-fermi-dirac.ipynb) (plus the Matsubara contours of [§7.2](complex-analysis-applications.ipynb)) must land on the same functions.

In [ ]:
# (solution hidden on the public site)


In [ ]:
# (solution hidden on the public site)


### Validation 2

In [ ]:
validate.close(
    E_deriv,
    E_closed,
    "⟨E⟩ = −∂lnZ/∂β = ħω(n_B + ½): the Planck occupation, by central differences",
    rtol=1e-8,
)
validate.close(
    planck_occupation(np.array([0.5, 1.0, 3.0])),
    1.0 / (np.exp(np.array([0.5, 1.0, 3.0])) - 1.0),
    "the expm1-safe occupation equals the textbook form where both are accurate",
    rtol=1e-12,
)

## Exercise 3 — Freezing out

The heat capacity's cliff — where equipartition dies, and the temperature scale that organizes
three notebooks. Cite {eq}`eq-freezing-out`.

1. Derive $C(T) = k_B\,x^2e^x/(e^x-1)^2$ from $\langle E\rangle$ and implement it
   `numpy.expm1`-safely (the `heat_capacity` helper).
2. Evaluate across $T/\theta_E = 0.1$ to $50$ and confirm the two regimes: $C/k_B = 0.0045$ at
   $0.1\,\theta_E$ (frozen) and $0.9997$ or better at $50\,\theta_E$ (classical).
3. Show the low-$T$ law is exponential, $C \approx k_B x^2 e^{-x}$, by fitting the slope of
   $\ln(C/x^2)$ against $x$ with `numpy.polyfit` (expect $-1$), and explain the gap's role: no
   thermal energy, no accessible excited state, no heat absorbed.
4. State the resolution (prose): Volume V's equipartition assigns $k_B$ per oscillator
   unconditionally; quantum discreteness switches oscillators *off* below $\theta_E$ — the
   volume's second "where classical failed" entry, and the switch that builds the staircase of [§7.6](molecules-rotation-vibration.ipynb).

In [ ]:
# (solution hidden on the public site)


In [ ]:
# (solution hidden on the public site)


### Validation 3

In [ ]:
validate.close(
    C_frozen,
    0.0045,
    "deep freeze at T = 0.1 θ_E: C/k_B ≈ 0.0045, the gap starving the oscillator",
    rtol=2e-2,
)
validate.close(
    C_classical,
    1.0,
    "equipartition recovered at T = 50 θ_E: the classical k_B per oscillator returns",
    rtol=1e-3,
)
validate.close(
    slope_fit,
    -1.0,
    "the frozen regime is exponential: the fitted slope of ln(C/x^2) vs x is −1 (Boltzmann)",
    rtol=1e-3,
)

## Exercise 4 — The classical limit derives h

Volume V divided phase space by $h$ on faith; the quantum high-temperature limit supplies the missing derivation —
and the *approach* to the limit is measured, not waved at. Cite {eq}`eq-classical-limit`.

1. Show $Z_q \to 1/(\beta\hbar\omega)$ at small $\beta\hbar\omega$ and verify the ratio
   $Z_q/(k_BT/\hbar\omega) \to 1$ numerically across $T/\theta_E = 1$ to $100$.
2. Evaluate Volume V's classical phase-space integral $\int dp\,dx\,e^{-\beta H}$ for the
   oscillator by two `scipy.integrate.quad` Gaussian factors, and note it has dimensions of
   *action* — it cannot be a state count as it stands.
3. Divide by $h = 2\pi\hbar$ and confirm the match to $Z_q$ at high $T$: the $1/h$ measure is the
   unique normalization making classical statistical mechanics the high-temperature limit of
   quantum statistical mechanics — Volume V's convention, derived.
4. Extract the approach to the limit: verify $(\langle E\rangle - k_BT)\cdot12k_BT/(\hbar\omega)^2
   \to 1$, identify the $(\hbar\omega)^2/12k_BT$ term, and name the Wigner $\hbar^2$ expansion it
   leads. (Prose + computation.)

In [ ]:
# (solution hidden on the public site)


### Validation 4

In [ ]:
validate.close(
    Zq_over_Zcl,
    1.0,
    "the 1/h phase-space measure is derived: Z_q meets (1/h)∫dp dx e^(−βH) at high T",
    rtol=1e-4,
)
validate.close(
    [wigner[2.0], wigner[20.0]],
    [1.0 - 1.0 / (60.0 * 4.0), 1.0],
    "the leading quantum correction (ħω)²/12k_BT, with the next Wigner term visible at T = 2",
    rtol=1e-2,
)
validate.check(
    ratio_high[1.0] < ratio_high[10.0] < ratio_high[100.0] < 1.0,
    "the approach to the classical count is monotone from below",
    f"ratios {ratio_high[1.0]:.4f} → {ratio_high[10.0]:.6f} → {ratio_high[100.0]:.6f}",
)

## Exercise 5 — The Einstein solid and the diamond anomaly

A nineteenth-century scandal, closed to three digits — and an honest failure that summons Debye.
Cite {eq}`eq-einstein-solid`.

1. Assemble $C_{\text{solid}}(T) = 3Nk_B\cdot x^2e^x/(e^x-1)^2$ with $x = \theta_E/T$ (the
   `heat_capacity` helper per mode) and confirm Dulong–Petit ($C \to 3Nk_B$, i.e. $24.9$
   J mol⁻¹K⁻¹) at high $T$.
2. With diamond's $\theta_E \approx 1320$ K, evaluate $C/3Nk_B$ at $300$ K and compare: predicted
   $0.244$ vs measured $\approx 0.245$ ($6.1$ of $24.9$ J mol⁻¹K⁻¹) — and explain *why* diamond
   (light atoms, stiff bonds → high $\omega_E$ → frozen at room temperature).
3. Evaluate the low-$T$ tail at $100$ K and $50$ K and state the failure: experiment shows
   $C \propto T^3$, not an exponential dive.
4. Diagnose in prose: independent equal-frequency oscillators share one gap; a real crystal's
   coupled modes include arbitrarily soft long-wavelength sound that no temperature can freeze —
   the physics Debye ([§7.16](phonons-debye.ipynb)) supplies.

In [ ]:
# (solution hidden on the public site)


In [ ]:
# (solution hidden on the public site)


### Validation 5

In [ ]:
validate.close(
    C_diamond_300,
    0.2436,
    "Einstein's resolution of the diamond anomaly: C/3Nk_B = 0.244 predicted at 300 K",
    rtol=1e-2,
)
validate.close(
    C_diamond_300,
    measured_300,
    "the prediction meets the measured 6.1/24.9 ≈ 0.245 to better than a percent",
    rtol=1e-2,
)
validate.check(
    C_50K < 1e-8 and C_100K < 1e-3 and C_high > 0.99,
    "the honest failure: an exponential low-T dive (vs the observed T³) under a correct plateau",
    f"C/3Nk_B = {C_100K:.1e} (100 K), {C_50K:.1e} (50 K)",
)

## Exercise 6 — Thermal photon statistics: the geometric distribution and bunching

The oscillator's number statistics, sampled two million times — and found guilty of bunching.
Cite {eq}`eq-photon-statistics`.

1. Derive $p_n = (1-e^{-x})e^{-nx}$ (geometric on $n \ge 0$) from the Boltzmann populations.
2. Sample $2\times10^6$ values with `numpy.random.default_rng().geometric(p = 1-e^{-x})`,
   applying the $n = k-1$ mapping (the sampler counts trials from $1$ — the off-by-one is stated
   in the `sample_photon_number` helper).
3. Verify the sampled mean against $\langle n\rangle$ and the sampled variance against
   $\langle\Delta n^2\rangle = \langle n\rangle(1+\langle n\rangle)$, and cross-check the
   variance against $\partial^2\ln Z/\partial\beta^2$ by central second differences.
4. Interpret (prose): the $+\langle n\rangle^2$ excess over Poisson is *bunching* — Einstein's
   1909 "wave" fluctuation term, the thermal-light signature Hanbury Brown and Twiss measured —
   and the contrast with Poissonian laser light is the business of [§7.15](einstein-a-b-coefficients.ipynb).

In [ ]:
# (solution hidden on the public site)


In [ ]:
# (solution hidden on the public site)


### Validation 6

In [ ]:
validate.close(
    mean_sampled,
    n_bar_exact,
    "the sampled thermal mean lands on the Planck occupation ⟨n⟩",
    rtol=1e-2,
)
validate.close(
    var_sampled,
    var_exact,
    "thermal light is super-Poissonian: ⟨Δn²⟩ = ⟨n⟩(1+⟨n⟩), sampled at 2e6 shots",
    rtol=1e-2,
)
validate.close(
    var_thermo,
    var_exact,
    "the thermodynamic route agrees: ∂²lnZ/∂β² = ⟨Δn²⟩ by central second differences",
    rtol=1e-5,
)

## Exercise 7 — The thermal width: the curve the path integral will resample

$\langle x^2\rangle(T)$ computed by trace machinery, from quantum floor to classical growth — and
flagged as a promise to a method four notebooks away. Cite {eq}`eq-thermal-width`.

1. Derive $\langle x^2\rangle = (\hbar/2m\omega)\coth(\beta\hbar\omega/2)$ from
   $\langle x^2\rangle_n = (n+\tfrac12)\hbar/m\omega$ averaged over the Boltzmann populations.
2. Build $a$ and $a^\dagger$ as matrices with `numpy.diag(numpy.sqrt(n), ±1)` (Volume VI's
   construction), form $x = \sqrt{\hbar/2m\omega}\,(a+a^\dagger)$ and $x^2$, and compute
   $\operatorname{Tr}(\rho\,x^2)$ in a truncated number basis ($n_{\max} = 400$; the
   `thermal_x2` helper).
3. Confirm trace and closed form agree at $\beta = 20, 1, 0.1$, and verify the two limits:
   $\hbar/2m\omega$ at low $T$ (the pure ground-state spread — quantum width that never freezes)
   and $k_BT/m\omega^2$ at high $T$ (classical equipartition of the potential term).
4. Flag the rendezvous (prose): this single curve is exactly what path-integral Monte Carlo
   ([§7.21](path-integral-monte-carlo.ipynb)) will recover by Metropolis sampling of classical ring polymers — the isomorphism's
   acid test, promised now.

In [ ]:
# (solution hidden on the public site)


In [ ]:
# (solution hidden on the public site)


### Validation 7

In [ ]:
validate.close(
    [x2_pairs[20.0][0], x2_pairs[1.0][0], x2_pairs[0.1][0]],
    [x2_pairs[20.0][1], x2_pairs[1.0][1], x2_pairs[0.1][1]],
    "⟨x²⟩ = (ħ/2mω)coth(βħω/2): the ladder-operator trace meets the closed form at all three β",
    rtol=1e-8,
)
validate.close(
    [x2_closed(20.0), x2_closed(0.1) / 10.0],
    [0.5, 1.0],
    "the two limits: the ground-state width at low T, classical equipartition k_BT/mω² at high T",
    rtol=1e-2,
)

## Exercise 8 — One oscillator, the whole volume in miniature

It is hard to overstate how much this single system contains. Its geometric-series partition
function produced the Bose function to partner the Fermi function of [§7.4](thermal-density-matrix.ipynb) — Movement I has now met both
quantum statistics one exactly-solvable system at a time, with the ensemble derivation of [§7.7](bose-einstein-fermi-dirac.ipynb) and
the contour route of [§7.2](complex-analysis-applications.ipynb) converging on the same pair. Its heat capacity died below a characteristic
temperature and resurrected equipartition above it, resolving the second classical failure on our
ledger with a single organizing scale, $\theta_E$. Its high-temperature limit reached back into
Volume V and derived the $h$ that classical mechanics had adopted on trust — and gave more than
was asked, since the approach to the limit carried the first Wigner correction, measured
converging like $1/T^2$. Scaled up by $3N$ it closed the diamond scandal to three digits and then
failed, honestly and instructively, in exactly the way that makes Debye necessary. Its number
fluctuations bunched, seeding the photon gas. And its thermal width traced the full arc from
quantum floor to classical spread — the curve the path integral will one day resample as its own
validation.

Notice what "classical limit" turned out to mean. Not a different theory taking over — just the
quantum one, viewed from far enough above its energy spacing that the rungs blur into a ramp. The
$h$ was there all along; classical mechanics simply could not see what it was normalizing by.
Planck quantized this oscillator to fit a spectrum; a century later it is still the clearest
window onto where the quantum world ends and the classical one begins. Next ([§7.6](molecules-rotation-vibration.ipynb)): molecules,
where several such switches (rotation, vibration) flip in sequence, and the heat capacity
climbs a staircase that nineteenth-century chemistry could measure but not explain.

## Notebook summary

The volume's founding calculation, exactly solvable and exactly verified.

- **The partition function is a geometric series** {eq}`eq-oscillator-Z`: $Z = 1/(2\sinh(\beta
  \hbar\omega/2))$, checked against truncated sums with the truncation rule demonstrated where it
  bites — at *high* temperature ($n_{\max} = 60$ fails the eight-digit standard at $\beta = 0.2$).
  $F \to \hbar\omega/2$ and $S \to 0$ at $T \to 0$: the third law on an infinite ladder.
- **The Planck occupation** {eq}`eq-planck-occupation`: $\langle E\rangle = \hbar\omega(\langle
  n\rangle + \tfrac12)$ with $\langle n\rangle = 1/(e^{\beta\hbar\omega}-1)$ — the Bose–Einstein
  function at $\mu = 0$, one movement early, because a bosonic mode *is* an oscillator.
  Movement I's symmetry is complete: the qubit gave $n_F$, the oscillator gives $n_B$.
- **Freezing out** {eq}`eq-freezing-out`: $C$ dives exponentially below $\theta_E$ (fitted slope
  $-1$ on the log-plot; $0.0045\,k_B$ at $0.1\,\theta_E$) and recovers equipartition above it —
  the second "where classical failed" resolution, and the switch that will build the staircase of [§7.6](molecules-rotation-vibration.ipynb).
- **The classical limit derives $h$** {eq}`eq-classical-limit`: Volume V's phase-space integral
  has dimensions of action and matches $Z_q$ at high $T$ only when divided by $h$ — the measure's
  $1/h$, derived; and the approach carries the $(\hbar\omega)^2/12k_BT$ Wigner correction,
  measured converging to its coefficient.
- **The Einstein solid** {eq}`eq-einstein-solid`: Dulong–Petit at high $T$; diamond's
  room-temperature "anomaly" closed to three digits ($0.244$ predicted vs $0.245$ measured at
  $\theta_E = 1320$ K); and the exponential low-$T$ tail failing against the observed $T^3$ —
  Debye's cue ([§7.16](phonons-debye.ipynb)), motivated by a wrong functional form rather than a wrong digit.
- **Thermal photon statistics** {eq}`eq-photon-statistics`: the geometric $p_n$, sampled
  $2\times10^6$ times (with the sampler's $n = k-1$ off-by-one stated), is super-Poissonian —
  $\langle\Delta n^2\rangle = \langle n\rangle(1+\langle n\rangle)$, bunching, Einstein's 1909
  wave term and the seed of [§7.14](photon-gas-planck.ipynb) — triangulated against $\partial^2\ln Z/\partial\beta^2$.
- **The thermal width** {eq}`eq-thermal-width`: $(\hbar/2m\omega)\coth(\beta\hbar\omega/2)$ by
  ladder-operator trace and closed form, from a quantum floor that never freezes to classical
  equipartition growth — the exact curve path-integral Monte Carlo ([§7.21](path-integral-monte-carlo.ipynb)) is now on record as
  owing us.

One oscillator; the volume in miniature.

## Outlook

- **Molecules ([§7.6](molecules-rotation-vibration.ipynb)).** The rotational and vibrational staircase built from today's freeze-out
  switch; ortho/para hydrogen.
- **The rendezvous ([§7.7](bose-einstein-fermi-dirac.ipynb)).** The grand-canonical derivation of $n_B$ and $n_F$, meeting the qubit,
  the oscillator, and the Matsubara contours of [§7.2](complex-analysis-applications.ipynb).
- **The payoffs.** The photon gas and Planck's law proper, with bunching and lasers ([§7.14](photon-gas-planck.ipynb), [§7.15](einstein-a-b-coefficients.ipynb));
  Debye's coupled modes and the $T^3$ law ([§7.16](phonons-debye.ipynb)); PIMC resampling the thermal width ([§7.21](path-integral-monte-carlo.ipynb)).
- **The Wigner $\hbar^2$ expansion** and semiclassical thermodynamics — a horizon, named.
- **Cross-reference** [§6.12](../06-quantum-mechanics/harmonic-oscillator.ipynb) (spectrum and ladder operators, reused), [§7.4](thermal-density-matrix.ipynb) (the canonical identities,
  the $n_F$ twin, the third law), [§7.2](complex-analysis-applications.ipynb) (the Matsubara route to the same $\coth$), [§5.5](../05-classical-stat-mech/ergodicity.ipynb)/[§5.6](../05-classical-stat-mech/ideal-gas-fundamental-relation.ipynb)
  (equipartition and the classical measure, now derived).

In [ ]:
from ecp.style import footer

footer()